In [101]:
import numpy as np
import pandas as pd
import random

import ollama
from sklearn.metrics import classification_report, balanced_accuracy_score

from pydantic import BaseModel, ValidationError
from typing import Literal, Optional

from pigeon import annotate
from IPython.display import display, HTML

from tqdm import tqdm
tqdm.pandas()

In [2]:
df = pd.read_csv("../data/articles/articles-v3.csv")

print(df.shape)
df.head()

(13480, 19)


,slug,company_name,title,link,published,source,status,city,county,state,category,lat,lon,content,final_url,facility_cluster,street_no,road,likely_relevant
0,vantage-ashburn-(va2)-data-center-campus,Vantage Data Centers,"as data center noise concerns grow, loudoun co...",https://news.google.com/rss/articles/CBMixwFBV...,2026-04-06 22:11:00,WUSA9,live,Sterling,Loudoun County,Virginia,Colocation,38.998031,-77.430415,"STERLING, Va. - For nine years, Lindsay Shaw e...",https://www.wusa9.com/article/news/local/data/...,1,22435,Glenn Drive,True
1,vantage-ashburn-(va2)-data-center-campus,Vantage Data Centers,a data center opened next door. then came the ...,https://news.google.com/rss/articles/CBMijwFBV...,2026-03-11 07:00:00,Politico,live,Sterling,Loudoun County,Virginia,Colocation,38.998031,-77.430415,A data center opened next door. Then came the ...,https://www.politico.com/news/2026/03/11/data-...,1,22435,Glenn Drive,True
2,vantage-ashburn-(va2)-data-center-campus,Vantage Data Centers,"microsoft? meta? no, it’s data center giant va...",https://news.google.com/rss/articles/CBMikwFBV...,2025-06-11 07:00:00,Ozaukee Press,live,Sterling,Loudoun County,Virginia,Colocation,38.998031,-77.430415,"Microsoft? Meta? No, its data center giant Van...",https://ozaukeepress.com/content/microsoft-met...,1,22435,Glenn Drive,True
3,vantage-ashburn-(va2)-data-center-campus,Vantage Data Centers,blackstone lands $550m for data center portfolio,https://news.google.com/rss/articles/CBMijwFBV...,2025-08-08 07:00:00,CommercialSearch,live,Sterling,Loudoun County,Virginia,Colocation,38.998031,-77.430415,Blackstone Lands $550M for Data Center Portfol...,https://www.commercialsearch.com/news/blacksto...,1,22435,Glenn Drive,True
4,vantage-ashburn-(va2)-data-center-campus,Vantage Data Centers,vantage data centers aggressively expands in n...,https://news.google.com/rss/articles/CBMibkFVX...,2022-05-23 07:00:00,Dgtl Infra,live,Sterling,Loudoun County,Virginia,Colocation,38.998031,-77.430415,"Vantage Data Centers, a hyperscale data center...",https://dgtlinfra.com/vantage-data-centers-nor...,1,22435,Glenn Drive,True


# Use LLM to Extract Event Data

## Setup Labeling Pipeline

In [43]:
PROMPT_TEMPLATE = """
You are a precise classifier for news articles about data center projects. You will be given a headline and the first portion of an article. Your job is to identify what lifecycle event (if any) the article reports about a specific named data center project.

Assign exactly ONE lifecycle_event label:

- proposal: First public disclosure of a specific new project at a specific site. Company announces plans, reveals site selection, files initial paperwork, or unveils a project for the first time.
- permitting: Regulatory activity on a named project — applications filed, hearings scheduled or held, votes taken, permits granted, permits denied (with project still alive), zoning changes, environmental review milestones.
- groundbreaking: Physical construction begins on the site (groundbreaking ceremony, site work starts, construction crews mobilize).
- operational: Facility opens, comes online, begins operations, or is "now serving" customers.
- expansion: An existing operational site is being made larger — new building, new phase, capacity increase. Use this regardless of which sub-stage the expansion is in.
- setback: A specific named project is paused, stalled, denied (still alive), under reconsideration, facing financing problems, or otherwise delayed without being killed.
- cancellation: A specific named project is confirmed dead at the original site — withdrawn, scrapped, abandoned, or relocated elsewhere.
- opposition: The article is primarily about community/local pushback, resident concerns, protests, organized opposition, or executive/official responses to such concerns regarding a specific named project — but the article does not report a concrete lifecycle movement (no vote, no permit decision, no cancellation, etc.). Use this when the news IS the opposition itself.
- other: Anything else. Industry news, market analysis, M&A of existing facilities, earnings, hiring, executive moves, hardware/chip news, opinion pieces, lawsuits or protests against unnamed projects, or any article without a specific identifiable project at a specific lifecycle event.

KEY PRINCIPLE: The article must report a specific lifecycle EVENT for a NAMED project at a SPECIFIC SITE. If either is missing, the label is "other".

DISAMBIGUATION RULES (apply in order, top rule wins):

1. M&A AND SALES OF EXISTING FACILITIES ARE "other". Buying, selling, or merging data center companies or already-built facilities is not a lifecycle event for a new project. A "proposal" requires a NEW project being announced for construction.

2. CANCELLATION > setback > lifecycle stage. If the project is confirmed dead at this site → cancellation. If only paused/stalled/denied-but-alive → setback. Otherwise use the appropriate lifecycle stage.

3. FIRST DISCLOSURE WINS. If the article is announcing a brand-new project for the first time, label "proposal" even if it mentions later milestones the company hopes to reach. Re-reporting of an already-announced project at the same stage is also "proposal" — deduplication happens downstream.

4. THE EVENT REPORTED WINS, NOT THE BACKGROUND. If the article's news is "Board approves project despite opposition," that is permitting (with local_opposition=true). If the news is "Lawsuit halts construction," that is setback. If the news is "residents speak out" with no concrete regulatory or project action, that is opposition. Identify what actually happened in this article, not what's been happening generally.

5. NAMED PROJECT REQUIREMENT. Industry-wide lawsuits, protests about hypothetical or unnamed sites, general activism, and policy debates all go to "other". The article must identify a specific project (company + location, or named campus). Opposition to a specific named project qualifies; generic opposition does not.

6. EXPANSION vs PROPOSAL. "expansion" requires the SAME site getting bigger. A new campus from the same company at a different site is "proposal".

7. OPPOSITION vs OTHER LABELS. If a concrete lifecycle event occurred (vote, permit, groundbreaking, cancellation, etc.), use that label and set local_opposition=true. Only use "opposition" when community pushback IS the news and no concrete lifecycle movement is reported. An executive defending a project against concerns, with no other action, is "opposition".

8. WHEN UNCERTAIN, DEFAULT TO "other". Do not guess.

LOCAL OPPOSITION ATTRIBUTE: Independently of lifecycle_event, set local_opposition=true if the article substantively features community pushback, resident concerns, organized protest, public outcry, demands for new conditions/mandates from local groups, or company/official responses to such pressure regarding the named project. Otherwise false. This attribute can be true alongside ANY lifecycle_event label.

OUTPUT FORMAT — respond with ONLY a JSON object, no other text:

{{
  "lifecycle_event": "<one of the labels above>",
  "permitting_outcome": "<filed | approved | denied | null>",
  "project_mentioned": <true | false>,
  "local_opposition": <true | false>
}}

EXAMPLES:

Input: "Meta unveils plans for $800M data center campus in Richland Parish, Louisiana. The social media giant announced Tuesday it will build a new hyperscale facility on a 2,400-acre site, with construction expected to begin in early 2026."
Output: {{ "lifecycle_event": "proposal", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "Loudoun County Board approves controversial Amazon data center despite resident objections. After a six-hour hearing, supervisors voted 6-3 to grant the rezoning needed for the 1.2 million square foot facility."
Output: {{ "lifecycle_event": "permitting", "permitting_outcome": "approved", "project_mentioned": true, "local_opposition": true }}

Input: "Microsoft pauses construction on Mount Pleasant data center expansion amid AI demand reassessment. The company said it is reconsidering the timeline for the second phase of its Wisconsin campus."
Output: {{ "lifecycle_event": "setback", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "Google scraps planned $200M Indiana data center, will not pursue site. The company confirmed Friday it has withdrawn its proposal for the Fort Wayne facility and has no plans to relocate the project to another Indiana site."
Output: {{ "lifecycle_event": "cancellation", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "QTS breaks ground on second Phoenix-area data center building. The company began site work Monday on the 450,000 square foot expansion to its existing Mesa campus."
Output: {{ "lifecycle_event": "expansion", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "Equinix opens new IBX data center in Frankfurt, expanding European capacity. The FR11 facility began serving customers this week."
Output: {{ "lifecycle_event": "operational", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "Village of DeForest approached about new data center. A developer presented preliminary plans to village trustees Monday for a potential data center on the north side of town."
Output: {{ "lifecycle_event": "proposal", "permitting_outcome": null, "project_mentioned": true, "local_opposition": false }}

Input: "'They want our land, they want our water': Caledonia residents speak out against planned data center. Dozens of residents packed Tuesday's town hall to voice concerns about the proposed Microsoft facility, though no vote was taken."
Output: {{ "lifecycle_event": "opposition", "permitting_outcome": null, "project_mentioned": true, "local_opposition": true }}

Input: "Former Texas Gov. Rick Perry addresses concerns over AI data center's water use. Speaking at a press event Wednesday, Perry sought to reassure Abilene residents about the Stargate project's water sourcing plan."
Output: {{ "lifecycle_event": "opposition", "permitting_outcome": null, "project_mentioned": true, "local_opposition": true }}

Input: "Neighbors of incoming data center in New Albany react to area growth. Residents living near the QTS campus shared mixed feelings about the development as construction continues."
Output: {{ "lifecycle_event": "opposition", "permitting_outcome": null, "project_mentioned": true, "local_opposition": true }}

Input: "Meta data center campus in Northland may face new MBE, WBE mandates after outcry. City council members are drafting requirements for minority and women-owned business participation following community demands."
Output: {{ "lifecycle_event": "permitting", "permitting_outcome": null, "project_mentioned": true, "local_opposition": true }}

Input: "Data center industry faces growing local opposition as residents organize against power demands. A new report finds community pushback has delayed dozens of projects nationwide."
Output: {{ "lifecycle_event": "other", "permitting_outcome": null, "project_mentioned": false, "local_opposition": false }}

Input: "Digital Realty acquires Teraco for $3.5 billion, expanding African footprint."
Output: {{ "lifecycle_event": "other", "permitting_outcome": null, "project_mentioned": false, "local_opposition": false }}

Now classify the following article.

Headline: {headline}

Article excerpt:
{article_excerpt}
"""

In [58]:
class ModelResponse(BaseModel):
    lifecycle_event: Literal["proposal", "permitting", "groundbreaking", "operational", "expansion", "setback", "cancellation", "opposition", "other"]
    permitting_outcome: Optional[Literal["filed", "approved", "denied"]] = None
    project_mentioned: bool
    local_opposition: bool

In [76]:
import time

def ollama_classify(row: pd.Series, max_content_length: int = 2500, num_tries: int = 3, model = "llama3.1:8b") -> pd.Series:
    content = row.get("content")
    excerpt = content[:max_content_length] if pd.notna(content) else ""
    prompt = PROMPT_TEMPLATE.format(headline=row["title"], article_excerpt=excerpt)

    last_error = None
    for attempt in range(1, num_tries + 1):
        try:
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                format="json"
            )["message"]["content"]

            data = ModelResponse.model_validate_json(response)
        except (ValidationError, KeyError, ollama.ResponseError) as e:
            last_error = e
            # print(f"Attempt {attempt}/{num_tries} failed: {type(e).__name__}: {e}")
            if attempt < num_tries:
                time.sleep(2 ** (attempt - 1))  # 1s, 2s, 4s...
            continue

        # Success — assign all fields atomically
        for field, value in data.model_dump().items():
            row[field] = value
        row["failed"] = False
        return row

    print(f"All {num_tries} attempts failed. Last error: {last_error}")
    row["lifecycle_event"] = None
    row["permitting_outcome"] = None
    row["project_mentioned"] = None
    row["local_opposition"] = None
    row["failed"] = True
    return row

## Manually Validate Using a Sample

In [155]:
# pick a 'random' case
random_slug = df["facility_cluster"].sample(1, random_state=42).item()
sample = df[df["facility_cluster"] == random_slug].sort_values("published", ascending=True)
sample

,slug,company_name,title,link,published,source,status,city,county,state,...,lon,content,final_url,facility_cluster,street_no,road,likely_relevant,llama_label,nli_label,nli_score
11871,google-lincoln-agate,Google,$600 million dollar data center project moving...,https://news.google.com/rss/articles/CBMilgFBV...,2019-07-24 07:00:00,"KOLN | Nebraska Local News, Weather, Sports | ...",under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,The silicon prairie in Lincoln may have the ch...,https://www.1011now.com/content/news/Massive-d...,1666,NaN,Cloud Court,True,approval,expansion,0.414173
11880,google-lincoln-agate,Google,lincoln city council approves nebraska data ce...,https://news.google.com/rss/articles/CBMiswFBV...,2019-09-13 07:00:00,Data Center Dynamics,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,"A $600 million data center project in Lincoln,...",https://www.datacenterdynamics.com/en/news/lin...,1666,NaN,Cloud Court,True,approval,proposal,0.386389
11889,google-lincoln-agate,Google,google to build $600 million data center in pa...,https://news.google.com/rss/articles/CBMi4wFBV...,2020-10-27 07:00:00,Lincoln Journal Star,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,Google on Friday announced plans to build a da...,https://journalstar.com/business/local/google-...,1666,NaN,Cloud Court,True,proposal,proposal,0.511701
11886,google-lincoln-agate,Google,google aims to help nebraska’s demand for cons...,https://news.google.com/rss/articles/CBMinwFBV...,2022-07-27 07:00:00,Nebraska Examiner,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,"20:54 Brief OMAHA - At least 100 teens, mostly...",https://nebraskaexaminer.com/briefs/google-aim...,1666,NaN,Cloud Court,True,other,construction,0.213326
11885,google-lincoln-agate,Google,prairie preservationists mourn loss of virgin ...,https://news.google.com/rss/articles/CBMivgFBV...,2022-09-06 07:00:00,Nebraska Examiner,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,5:00 News Story Prairie preservationists mourn...,https://nebraskaexaminer.com/2022/09/06/prairi...,1666,NaN,Cloud Court,True,opposition,opposition,0.260668
11861,google-lincoln-agate,Google,"work on data center in lincoln, nebraska final...",https://news.google.com/rss/articles/CBMiogFBV...,2023-06-07 07:00:00,Data Center Dynamics,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,Work on a data center campus first put forward...,https://www.datacenterdynamics.com/en/news/wor...,1666,NaN,Cloud Court,True,construction,construction,0.507558
11888,google-lincoln-agate,Google,power-hungry data centers help drive need for ...,https://news.google.com/rss/articles/CBMi5AFBV...,2023-07-30 07:00:00,Omaha World-Herald,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,A pair of Facebook data center complexes strad...,https://omaha.com/news/local/power-hungry-data...,1666,NaN,Cloud Court,False,expansion,expansion,0.408188
11865,google-lincoln-agate,Google,google investing another $1.2 billion in papil...,https://news.google.com/rss/articles/CBMioAFBV...,2023-08-22 07:00:00,WOWT,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,"OMAHA, Neb. (WOWT) - Google is making another ...",https://www.wowt.com/2023/08/22/google-investi...,1666,NaN,Cloud Court,True,approval,proposal,0.426559
11864,google-lincoln-agate,Google,google data center to be built in north lincoln,https://news.google.com/rss/articles/CBMiggFBV...,2023-08-22 07:00:00,klin.com,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,Google has announced plans to build a data cen...,https://klin.com/2023/08/22/google-data-center...,1666,NaN,Cloud Court,True,proposal,proposal,0.464482
11863,google-lincoln-agate,Google,"google announces new lincoln data center, addi...",https://news.google.com/rss/articles/CBMixAFBV...,2023-08-22 07:00:00,KMTV 3 News Now,under_construction,Lincoln,Lancaster County,Nebraska,...,-96.649971,

In [119]:
def display_fn(row):
    title = row["title"]
    content = row["content"]

    if pd.isna(content):
        body = ""
    else:
        body = str(content)[:150]

    display(HTML(f"<h4>{title}</h4><p>{body}</p>"))

annotations = annotate(
    sample.to_dict("records"),
    options=["proposal", "permitting", "groundbreaking", "operational", "expansion", "setback", "cancellation", "opposition", "other"],
    include_skip=False,
    shuffle=False,
    display_fn=display_fn,
)

HTML(value='0 examples annotated, 9 examples left')

Dropdown(options=('proposal', 'permitting', 'groundbreaking', 'operational', 'expansion', 'setback', 'cancella…

Output()

### `llama3.1:8b`

In [120]:
llama_sample = sample.copy().progress_apply(lambda row: ollama_classify(row, model="llama3.1:8b"), axis=1, result_type="expand")
llama_sample[["title", "lifecycle_event", "permitting_outcome", "project_mentioned", "local_opposition", "failed", "final_url"]]

100%|██████████| 8/8 [00:22<00:00,  2.75s/it]


,title,lifecycle_event,permitting_outcome,project_mentioned,local_opposition,failed,final_url
11008,ne edge proposes 1.5 million sq ft data center...,proposal,None,True,False,False,https://www.datacenterdynamics.com/en/news/ne-...
11007,shoreline residents concerned with plan for la...,opposition,None,True,True,False,https://www.wfsb.com/2024/01/18/shoreline-resi...
11010,proposed data center would get power from mill...,proposal,None,True,False,False,https://hartfordbusiness.com/article/proposed-...
11009,plans for massive data center linked to nuclea...,opposition,None,True,True,False,https://ctexaminer.com/2024/04/08/plans-for-ma...
11004,don’t let a hyper-data center suck up ct’s ele...,other,None,False,False,False,https://ctmirror.org/2024/04/19/ct-data-center...
11005,waterford neighbors opposing proposed data cen...,opposition,None,True,True,False,https://www.nbcconnecticut.com/news/local/wate...
11006,data centers could transform local ct communit...,other,None,False,False,False,https://www.ctinsider.com/business/article/eff...
11011,nuclear power plant waste facilities closing,other,None,False,False,False,https://insideinvestigator.org/nuclear-power-p...


In [121]:
# evaluate against manual annotations
y_preds_llama = llama_sample["lifecycle_event"].tolist()
y_true = [i[1] for i in annotations]

print(classification_report(y_true, y_preds_llama))

              precision    recall  f1-score   support

  opposition       1.00      1.00      1.00         3
       other       0.67      0.67      0.67         3
    proposal       0.50      1.00      0.67         1
     setback       0.00      0.00      0.00         1

    accuracy                           0.75         8
   macro avg       0.54      0.67      0.58         8
weighted avg       0.69      0.75      0.71         8



C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### `phi3:mini`

In [122]:
phi_sample = sample.copy().progress_apply(lambda row: ollama_classify(row, model="phi3:mini"), axis=1, result_type="expand")
phi_sample[["title", "lifecycle_event", "permitting_outcome", "project_mentioned", "local_opposition", "failed", "final_url"]]

100%|██████████| 8/8 [00:18<00:00,  2.37s/it]


,title,lifecycle_event,permitting_outcome,project_mentioned,local_opposition,failed,final_url
11008,ne edge proposes 1.5 million sq ft data center...,proposal,None,True,False,False,https://www.datacenterdynamics.com/en/news/ne-...
11007,shoreline residents concerned with plan for la...,opposition,None,True,True,False,https://www.wfsb.com/2024/01/18/shoreline-resi...
11010,proposed data center would get power from mill...,proposal,None,True,False,False,https://hartfordbusiness.com/article/proposed-...
11009,plans for massive data center linked to nuclea...,proposal,None,True,False,False,https://ctexaminer.com/2024/04/08/plans-for-ma...
11004,don’t let a hyper-data center suck up ct’s ele...,proposal,None,True,False,False,https://ctmirror.org/2024/04/19/ct-data-center...
11005,waterford neighbors opposing proposed data cen...,opposition,None,True,True,False,https://www.nbcconnecticut.com/news/local/wate...
11006,data centers could transform local ct communit...,other,None,False,True,False,https://www.ctinsider.com/business/article/eff...
11011,nuclear power plant waste facilities closing,other,None,False,False,False,https://insideinvestigator.org/nuclear-power-p...


In [123]:
# evaluate against manual annotations
y_preds_phi = phi_sample["lifecycle_event"].tolist()
y_true = [i[1] for i in annotations]

print(classification_report(y_true, y_preds_phi))

              precision    recall  f1-score   support

  opposition       1.00      0.67      0.80         3
       other       0.50      0.33      0.40         3
    proposal       0.25      1.00      0.40         1
     setback       0.00      0.00      0.00         1

    accuracy                           0.50         8
   macro avg       0.44      0.50      0.40         8
weighted avg       0.59      0.50      0.50         8



C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\miniforge3\envs\data-centers\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Label Full Dataset

# stratified validation

- take a stratified sample of 200 headlines.
- manually label using pigeon
- validate against output

# Finetune Bert

# coherence checks

For each project_id, look at the sequence of (date, lifecycle_event) pairs. A coherent project looks like proposal → permitting → groundbreaking → operational, possibly with setbacks interspersed. Projects with operational before proposal, or multiple cancellation events followed by more activity, are flagging either classifier errors or noisy project_ids. Counting the rate of incoherent sequences is a quasi-validation metric — if it goes from 18% to 6% after a prompt change, the prompt change probably helped.

Setback rate sanity. Industry reporting suggests something like 15–25% of announced data center projects experience public setbacks. If your classifier produces setback on 5% or 40% of project-articles, something's off in either direction.

Time-to-event distributions. Once you've got labels, plot the distribution of days from proposal to groundbreaking per project. If it's bimodal with a spike at zero, you're probably mislabeling re-reports as new proposals. If the median is wildly off published industry benchmarks (~18–30 months for hyperscale), classification errors are likely smearing the signal.